<a href="https://colab.research.google.com/github/islam37/S-SLTE-Thesis/blob/main/01_dataset_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -q datasets pandas scikit-learn


In [ ]:

from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

import os

DATA_DIR = "/content/drive/MyDrive/S_SLTE_Thesis/data"

os.makedirs(DATA_DIR, exist_ok=True)

print("Folder created:", DATA_DIR)


Folder created: /content/drive/MyDrive/S_SLTE_Thesis/data


In [ ]:

from datasets import load_dataset

# Download AG News
dataset = load_dataset("fancyzhx/ag_news")

# Show dataset information
print(dataset)


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [ ]:

from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

import os

DATA_DIR = "/content/drive/MyDrive/S_SLTE_Thesis/data"

os.makedirs(DATA_DIR, exist_ok=True)

print("Folder ready:", DATA_DIR)


Folder ready: /content/drive/MyDrive/S_SLTE_Thesis/data


In [ ]:

# Save original AG News dataset

train_path = f"{DATA_DIR}/ag_news_train.csv"
test_path = f"{DATA_DIR}/ag_news_test.csv"

dataset["train"].to_csv(train_path)
dataset["test"].to_csv(test_path)

print("Training dataset saved:", train_path)
print("Test dataset saved:", test_path)
print("Dataset backup completed!")


Creating CSV from Arrow format:   0%|          | 0/120 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/8 [00:00<?, ?ba/s]

Training dataset saved: /content/drive/MyDrive/S_SLTE_Thesis/data/ag_news_train.csv
Test dataset saved: /content/drive/MyDrive/S_SLTE_Thesis/data/ag_news_test.csv
Dataset backup completed!


In [ ]:

import pandas as pd

train_df = pd.read_csv(
    f"{DATA_DIR}/ag_news_train.csv"
)

test_df = pd.read_csv(
    f"{DATA_DIR}/ag_news_test.csv"
)

print("Training:", len(train_df))
print("Test:", len(test_df))

print("\nFirst 5 News:")
display(train_df.head())


Training: 120000
Test: 7600

First 5 News:


,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [ ]:

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42

# Original dataset row numbers
all_ids = np.arange(len(train_df))
all_labels = train_df["label"].to_numpy()

# Select 20,000 examples proportionally
selected_ids, _ = train_test_split(
    all_ids,
    train_size=20000,
    stratify=all_labels,
    random_state=SEED
)

# 2,000 clean anchor examples
anchor_ids, remaining_ids = train_test_split(
    selected_ids,
    train_size=2000,
    stratify=all_labels[selected_ids],
    random_state=SEED
)

# Split remaining 18,000 examples
# into 1,800 development and 16,200 training
dev_ids, noisy_ids = train_test_split(
    remaining_ids,
    train_size=1800,
    stratify=all_labels[remaining_ids],
    random_state=SEED
)

print("Clean Anchor:", len(anchor_ids))
print("Development:", len(dev_ids))
print("Noisy Training:", len(noisy_ids))
print("Total:", len(selected_ids))

# Ensure no shared record IDs
assert len(set(anchor_ids) & set(dev_ids)) == 0
assert len(set(anchor_ids) & set(noisy_ids)) == 0
assert len(set(dev_ids) & set(noisy_ids)) == 0

print("All splits are disjoint!")


Clean Anchor: 2000
Development: 1800
Noisy Training: 16200
Total: 20000
All splits are disjoint!


In [ ]:

def prepare_dataframe(ids):
    result = train_df.iloc[sorted(ids)].copy()

    # Preserve original AG News row number
    result.insert(0, "record_id", sorted(ids))

    return result[["record_id", "text", "label"]]

anchor_df = prepare_dataframe(anchor_ids)
dev_df = prepare_dataframe(dev_ids)
noisy_reference_df = prepare_dataframe(noisy_ids)

# Hide reference labels from the LLM training pool
noisy_df = noisy_reference_df[
    ["record_id", "text"]
].copy()

print("Anchor:", anchor_df.shape)
print("Development:", dev_df.shape)
print("Noisy:", noisy_df.shape)

display(noisy_df.head())


Anchor: (2000, 3)
Development: (1800, 3)
Noisy: (16200, 2)


,record_id,text
11,11,No Need for OPEC to Pump More-Iran Gov TEHRAN...
14,14,Dollar Falls Broadly on Record Trade Gap NEW ...
19,19,Shell 'could be target for Total' Oil giant Sh...
43,43,Grant to aid Lynn Central Square Central Squar...
44,44,State grant to aid Lynn; Bank gives Salem \$10...


In [ ]:

import os

SPLIT_DIR = f"{DATA_DIR}/splits"

PRIVATE_DIR = (
    "/content/drive/MyDrive/"
    "S_SLTE_Thesis/private_reference"
)

os.makedirs(SPLIT_DIR, exist_ok=True)
os.makedirs(PRIVATE_DIR, exist_ok=True)

anchor_df.to_csv(
    f"{SPLIT_DIR}/anchor.csv", index=False
)

dev_df.to_csv(
    f"{SPLIT_DIR}/development.csv", index=False
)

noisy_df.to_csv(
    f"{SPLIT_DIR}/noisy_training.csv", index=False
)

# Private reference, for retrospective evaluation
noisy_reference_df.to_csv(
    f"{PRIVATE_DIR}/noisy_reference.csv",
    index=False
)

print("All research splits saved!")


All research splits saved!


In [ ]:

categories = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

for name, data in [
    ("Anchor", anchor_df),
    ("Development", dev_df),
    ("Noisy Reference", noisy_reference_df)
]:
    print(f"\n{name}:")
    print(
        data["label"]
        .map(categories)
        .value_counts()
        .sort_index()
    )



Anchor:
label
Business    500
Sci/Tech    500
Sports      500
World       500
Name: count, dtype: int64

Development:
label
Business    450
Sci/Tech    450
Sports      450
World       450
Name: count, dtype: int64

Noisy Reference:
label
Business    4050
Sci/Tech    4050
Sports      4050
World       4050
Name: count, dtype: int64


In [ ]:

from google.colab import drive
import os
import pandas as pd

# Connect Google Drive
drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/S_SLTE_Thesis"

SPLIT_DIR = f"{BASE_DIR}/data/splits"
PRIVATE_DIR = f"{BASE_DIR}/private_reference"

print("Google Drive connected!")



# Expected research files
paths = {
    "anchor": f"{SPLIT_DIR}/anchor.csv",
    "development": f"{SPLIT_DIR}/development.csv",
    "noisy": f"{SPLIT_DIR}/noisy_training.csv",
    "reference": f"{PRIVATE_DIR}/noisy_reference.csv"
}

# Ensure all files exist
for name, path in paths.items():
    assert os.path.exists(path), f"Missing file: {path}"

# Load saved splits
anchor_df = pd.read_csv(paths["anchor"])
dev_df = pd.read_csv(paths["development"])
noisy_df = pd.read_csv(paths["noisy"])

# Check dataset sizes
assert len(anchor_df) == 2000
assert len(dev_df) == 1800
assert len(noisy_df) == 16200

# Check that record IDs are unique
for name, df in [
    ("Anchor", anchor_df),
    ("Development", dev_df),
    ("Noisy", noisy_df)
]:
    assert df["record_id"].is_unique
    assert df["text"].notna().all()

# Check partitions do not overlap
anchor_ids = set(anchor_df["record_id"])
dev_ids = set(dev_df["record_id"])
noisy_ids = set(noisy_df["record_id"])

assert anchor_ids.isdisjoint(dev_ids)
assert anchor_ids.isdisjoint(noisy_ids)
assert dev_ids.isdisjoint(noisy_ids)

# Verify label visibility
assert "label" in anchor_df.columns
assert "label" in dev_df.columns
assert "label" not in noisy_df.columns

print("DATASET VERIFICATION PASSED")
print("Clean Anchor:", len(anchor_df))
print("Development:", len(dev_df))
print("Noisy Training:", len(noisy_df))
print("All partitions are disjoint.")
print("Noisy-training reference labels are hidden.")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive connected!
DATASET VERIFICATION PASSED
Clean Anchor: 2000
Development: 1800
Noisy Training: 16200
All partitions are disjoint.
Noisy-training reference labels are hidden.


In [3]:

# Reproducible pilot sampling
PILOT_SEED = 42

pilot_df = noisy_df.sample(
    n=500,
    random_state=PILOT_SEED
).sort_values("record_id").reset_index(drop=True)

# Check pilot integrity
assert len(pilot_df) == 500
assert pilot_df["record_id"].is_unique
assert "label" not in pilot_df.columns
assert set(pilot_df["record_id"]).issubset(noisy_ids)

print("PILOT DATASET CREATED")
print("Total News:", len(pilot_df))
print("Columns:", pilot_df.columns.tolist())

display(pilot_df.head())


PILOT DATASET CREATED
Total News: 500
Columns: ['record_id', 'text']


,record_id,text
0,43,Grant to aid Lynn Central Square Central Squar...
1,188,Ashlee Vance: the readers have spoken &lt;stro...
2,765,Stewart spoils upset bid Sports car ace Ron Fe...
3,913,"Haas, Cink Selected for Ryder Cup Team MILWAUK..."
4,926,Singer 'may move to French jail' French rock s...


In [4]:

PILOT_DIR = f"{BASE_DIR}/data/pilot"
os.makedirs(PILOT_DIR, exist_ok=True)

pilot_path = f"{PILOT_DIR}/pilot_500.csv"

# Avoid silently changing an existing pilot
if os.path.exists(pilot_path):
    previous = pd.read_csv(pilot_path)
    pd.testing.assert_frame_equal(previous, pilot_df)
    print("Existing pilot matches exactly.")
else:
    pilot_df.to_csv(
        pilot_path,
        index=False,
        encoding="utf-8"
    )
    print("New pilot saved.")

# Verify saved file
saved_pilot = pd.read_csv(pilot_path)

pd.testing.assert_frame_equal(
    saved_pilot,
    pilot_df
)

print("PILOT SAVE VERIFIED")
print("Records:", len(saved_pilot))
print("Location:", pilot_path)


New pilot saved.
PILOT SAVE VERIFIED
Records: 500
Location: /content/drive/MyDrive/S_SLTE_Thesis/data/pilot/pilot_500.csv
